# Text-to-SQL with LlamaIndex: query engines over Postgres

**Stack:** LlamaIndex (`llama-index-core`) · Neon serverless Postgres · OpenAI

Where the LangChain notebook *built an agent loop by hand*, this one shows the
**query-engine** philosophy: LlamaIndex packages "retrieve context -> synthesize SQL
-> execute -> answer" into composable objects you configure rather than code.

The progression mirrors how the framework's own docs teach it:

1. `NLSQLTableQueryEngine` — the baseline: hand it the tables, it does the rest.
2. `SQLTableRetrieverQueryEngine` — the scale fix: **retrieve relevant table schemas
   per question** instead of stuffing every DDL into the prompt (the semantic layer
   lives in `context_str` per table).
3. `rows_retrievers` / `cols_retrievers` — the entity-grounding fix: teach the model
   your *vocabulary*, not just your schema ("US" vs "United States").
4. `NLSQLRetriever` — the primitives, composed by hand, so you see what the packaged
   engines were doing all along.

> **Security, up front.** LlamaIndex had CVE-2024-23751: natural-language input like
> "drop the students table" was executed verbatim by the text-to-SQL engines.
> Mitigations: connect as the read-only `text2sql_ro` role from `scripts/schema.sql`,
> and treat generated SQL as untrusted regardless of framework defaults. LlamaIndex's
> packaged engines do not validate SQL the way our LangChain guard does — that is a
> real, deliberate trade-off you are about to see.

## 0. Setup and connection

Same Neon engine rules as the LangChain notebook: pooled hostname, psycopg 3 scheme,
`pool_pre_ping` for suspend/restart resilience. We then wrap the engine in LlamaIndex's
`SQLDatabase` — a thin SQLAlchemy facade the query engines consume.

## 0a. Connect as the read-only role — non-negotiable

While testing this exact notebook, the packaged engines **executed DELETE statements and
dropped the `reviews` table** — the LLM wrote destructive SQL in response to eval
questions, and the engine ran it against the database because it held owner credentials.
That is CVE-2024-23751, reproduced on a live database. The lesson generalizes:

> Any component that executes model-generated SQL must run under credentials that
> cannot modify data — regardless of what the prompt says.

`scripts/schema.sql` creates `text2sql_ro` (SELECT-only, `default_transaction_read_only`).
We derive its connection string from your `DATABASE_URL` here, so the worst any generated
query can do is fail with a permission error.

In [1]:
import os
import re

from dotenv import load_dotenv
from llama_index.core import SQLDatabase
from llama_index.llms.openai import OpenAI
from sqlalchemy import create_engine, text

load_dotenv()

raw_url = os.environ["DATABASE_URL"]
DATABASE_URL = raw_url if "+psycopg" in raw_url else raw_url.replace(
    "postgresql://", "postgresql+psycopg://", 1)
assert "-pooler" in DATABASE_URL and "sslmode=require" in DATABASE_URL

# Swap owner credentials for the read-only role created by scripts/schema.sql.
# Contrast with notebook 01, where the guard + RO role are layered defenses; here the
# role IS the only defense, because the packaged engines execute generated SQL verbatim.
DATABASE_URL_RO = re.sub(
    r"(postgresql(?:\+psycopg)?://)[^@]+@",
    r"\g<1>text2sql_ro:Text2sql_ro!2026@",
    DATABASE_URL, count=1)

engine = create_engine(
    DATABASE_URL_RO,
    pool_size=5,
    pool_pre_ping=True,   # Neon computes suspend; pre_ping avoids stale-connection errors
    pool_recycle=300,
)

llm = OpenAI(model=os.getenv("LLM_MODEL", "gpt-4.1-mini"), temperature=0.1)
# temperature 0.1: SQL generation is a precision task; keep the model near-deterministic

with engine.connect() as conn:
    print("orders rows:", conn.execute(text("SELECT COUNT(*) FROM orders")).scalar())

sql_database = SQLDatabase(engine, schema="public")
print("\nLlamaIndex sees these tables:", sql_database.get_usable_table_names())

orders rows: 40000



LlamaIndex sees these tables: ['addresses', 'categories', 'customers', 'order_items', 'orders', 'payments', 'products', 'reviews', 'shipments', 'suppliers']


## 0b. Instrumentation: OpenInference spans -> Langfuse

LlamaIndex has no Langfuse-native callback in current versions. The documented path is
[OpenInference](https://docs.arize.com/phoenix/tracing/integrations-tracing/llamaindex):
its `LlamaIndexInstrumentor` emits every LlamaIndex operation as OpenTelemetry spans, and
Langfuse ingests OTel directly at `/api/public/otel`. We point the OTLP exporter at
Langfuse and build the auth header from your keys — no extra secrets needed.

Architecture worth teaching: **one instrumentation standard (OTel), many backends.**
Swap Langfuse for a self-hosted collector later by changing two env vars, not your app.

In [2]:
import base64

from langfuse import get_client, propagate_attributes
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor

# Route OpenInference spans to Langfuse (must be set BEFORE instrumenting):
# - endpoint: Langfuse's OTLP endpoint
# - headers: Basic auth (base64 public_key:secret_key) + the v4 ingestion header
#   so traces appear in real time (without it: up to ~10 min delay)
auth = base64.b64encode(
    f"{os.environ['LANGFUSE_PUBLIC_KEY']}:{os.environ['LANGFUSE_SECRET_KEY']}".encode()
).decode()
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT",
                      os.environ["LANGFUSE_BASE_URL"].rstrip("/") + "/api/public/otel")
os.environ.setdefault("OTEL_EXPORTER_OTLP_HEADERS",
                      f"Authorization=Basic {auth},x-langfuse-ingestion-version=4")

LlamaIndexInstrumentor().instrument()

langfuse = get_client()
assert langfuse.auth_check(), "Langfuse auth failed - check LANGFUSE_* in .env"
print("LlamaIndex instrumented; spans will land in Langfuse.")

LlamaIndex instrumented; spans will land in Langfuse.


## 1. Baseline: `NLSQLTableQueryEngine`

The simplest path. You tell it which tables may be used; it pulls those schemas into
the prompt, asks the LLM for SQL, executes, and synthesizes a natural-language answer.

**Key discipline — `include_tables` / `tables`.** Without it, the engine serializes
*every* table's DDL into the prompt. With 10 tables × ~15 columns that is already
hundreds of tokens of noise; at 100 tables it overflows the context window and quality
collapses. Grounding the model in *relevant* schema only is the single biggest
accuracy lever in text-to-SQL.

In [3]:
from llama_index.core.query_engine import NLSQLTableQueryEngine

query_engine = NLSQLTableQueryEngine(
    sql_database=sql_database,
    tables=["orders", "order_items", "products", "categories"],  # allow-list!
    llm=llm,
)

response = query_engine.query("Which product category generated the most revenue?")
print(response)
print("\n--- what actually happened ---")
print("SQL:   ", response.metadata["sql_query"])
print("Result:", response.metadata["result"])

The product category that generated the most revenue is "Hope Lab 15713," with a total revenue of $67,025.66.

--- what actually happened ---
SQL:    SELECT categories.name AS category_name, 
       SUM(order_items.quantity * order_items.unit_price) AS revenue
FROM order_items
JOIN orders ON order_items.order_id = orders.order_id
JOIN products ON order_items.product_id = products.product_id
JOIN categories ON products.category_id = categories.category_id
WHERE orders.status != 'cancelled'
GROUP BY categories.name
ORDER BY revenue DESC
LIMIT 1;
Result: [('Hope Lab 15713', Decimal('67025.66'))]


## 2. Scale: `SQLTableRetrieverQueryEngine` + a semantic layer from the DB itself

The baseline forces *you* to choose tables per question. The retriever variant inverts
that: embed each table's schema **with a human-written description** (`context_str`),
index them, and retrieve the top-k relevant tables *at query time*.

Our schema already carries that semantic layer — every table and column has a
`COMMENT ON` from `scripts/schema.sql`. This cell pulls those comments straight out of
`pg_catalog` and builds `SQLTableSchema` objects from them. **Maintain the semantic
layer in the database, where it lives next to the data it describes** — not in a
Python file someone forgets to update.

In [4]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.indices.struct_store.sql_query import SQLTableRetrieverQueryEngine
from llama_index.core.objects import ObjectIndex, SQLTableNodeMapping, SQLTableSchema

TABLES = ["categories", "suppliers", "customers", "addresses", "products",
          "orders", "order_items", "payments", "shipments", "reviews"]


def table_context(table: str) -> str:
    """Compose a context_str for one table from its live COMMENT metadata."""
    with engine.connect() as conn:
        t_comment = conn.execute(text(
            "SELECT obj_description(to_regclass(:t))"), {"t": f"public.{table}"}).scalar()
        cols = conn.execute(text("""
            SELECT a.attname, col_description(a.attrelid, a.attnum)
            FROM pg_attribute a
            JOIN pg_class c ON c.oid = a.attrelid
            WHERE c.relname = :t AND a.attnum > 0 AND NOT a.attisdropped
              AND col_description(a.attrelid, a.attnum) IS NOT NULL
        """), {"t": table}).all()
    col_text = "; ".join(f"{name}: {desc}" for name, desc in cols)
    return f"{t_comment or ''}\nColumn notes: {col_text}"


table_schema_objs = [SQLTableSchema(table_name=t, context_str=table_context(t))
                     for t in TABLES]

table_node_mapping = SQLTableNodeMapping(sql_database)
obj_index = ObjectIndex.from_objects(
    table_schema_objs,
    table_node_mapping,
    VectorStoreIndex,
    embed_model=OpenAIEmbedding(model="text-embedding-3-small"),
)

retriever_engine = SQLTableRetrieverQueryEngine(
    sql_database,
    obj_index.as_retriever(similarity_top_k=2),  # only 2 most-relevant schemas per question
    llm=llm,
)

response = retriever_engine.query("Which customers never placed any order?")
print(response)
print("\nSQL used:", response.metadata["sql_query"])

The customers who never placed any order are those whose customer IDs and names appear in the query result where there is no matching order record. According to the SQL query and the response data, these customers include (but are not limited to):

- Natasha Terry (Customer ID 5)
- Eric Webster (Customer ID 8)
- Kevin Romero (Customer ID 16)
- Daniel Travis (Customer ID 18)
- Joseph Garrett (Customer ID 19)
- Joshua Mcguire (Customer ID 25)
- Matthew Cook (Customer ID 49)
- Teresa Brewer (Customer ID 59)
- Michelle Kerr (Customer ID 68)
- Frank White (Customer ID 86)
- Larry Hart (Customer ID 87)
- Wendy Gutierrez (Customer ID 93)
- Danielle Barker (Customer ID 114)
- Antonio Singh (Customer ID 118)
- Craig Freeman (Customer ID 121)
- Shane Murphy (Customer ID 130)
- Selena Pearson (Customer ID 134)
- Guy Snyder (Customer ID 141)
- Ryan Sims (Customer ID 146)
- Laura Hill (Customer ID 152)
- John Walton (Customer ID 154)
- Eric Snyder (Customer ID 156)
- Melanie Sampson (Customer ID 15

## 3. Entity grounding: `rows_retrievers` and `cols_retrievers`

Schemas tell the model *structure*; they don't tell it *vocabulary*. Ask
"how many orders shipped to the US?" and a model may generate
`WHERE country = 'US'` — but our data says `'United States'`, and the count comes
back zero. This is the most common silent failure in text-to-SQL: **the query is
syntactically perfect and semantically empty.**

The fix: embed the *values* of categorical columns and inject the closest matches
into the prompt at query time. `cols_retrievers` indexes distinct values per column
(cheaper than embedding every row); `rows_retrievers` embeds full rows for
fine-grained cases. We demonstrate with `customers.country` and `orders.status` —
run the two queries below and compare counts.

In [5]:
from llama_index.core.schema import TextNode

# The engine looks up cols_retrievers[table] for EVERY retrieved table, so every
# table in the ObjectIndex needs an entry; tables without a designated column
# get an empty dict (no value context injected). This KeyError is a classic
# first-run failure when you index values for only some tables.
COL_VALUE_COLUMNS = {"customers": ["country"], "orders": ["status"]}
cols_retrievers = {t: {} for t in TABLES}
for table, columns in COL_VALUE_COLUMNS.items():
    for column in columns:
        with engine.connect() as conn:
            values = conn.execute(text(f'SELECT DISTINCT "{column}" FROM public."{table}"')).all()
        nodes = [TextNode(text=v[0]) for v in values]
        idx = VectorStoreIndex(nodes, embed_model=OpenAIEmbedding(model="text-embedding-3-small"))
        cols_retrievers[table][column] = idx.as_retriever(similarity_top_k=3)

grounded_engine = SQLTableRetrieverQueryEngine(
    sql_database,
    obj_index.as_retriever(similarity_top_k=2),
    cols_retrievers=cols_retrievers,
    llm=llm,
)

# "US" must map to "United States" via the column-value retriever:
with langfuse.start_as_current_observation(as_type="span", name="text2sql-query") as span:
    span.update(input={"question": "How many customers are in the US?"})
    with propagate_attributes(session_id="li-demo", tags=["demo", "llamaindex"]):
        r = grounded_engine.query("How many customers are in the US?")
    print("US  ->", r, "| SQL:", r.metadata["sql_query"])

langfuse.flush()

US  -> There are 2,463 customers in the United States. | SQL: SELECT COUNT(*) FROM customers WHERE country = 'United States';


## 4. Under the hood: `NLSQLRetriever` composed by hand

Every engine above is just a retriever + a response synthesizer. Stripping the
packaging away shows the moving parts — useful when you need custom behaviour
(e.g. streaming, custom prompts, or routing questions between SQL and other indices):

In [6]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import NLSQLRetriever

# return_raw=True: nodes contain the raw result rows; False: one node per row dict
nl_sql_retriever = NLSQLRetriever(
    sql_database, tables=["orders", "customers"], llm=llm, return_raw=True,
)
query_engine = RetrieverQueryEngine.from_args(nl_sql_retriever, llm=llm)

response = query_engine.query(
    "What are the top 3 countries by number of orders, with counts?")
print(response)

The top 3 countries by number of orders are France with 5174 orders, United Kingdom with 5113 orders, and United States with 5055 orders.


## 5. Adversarial input — same tests as the LangChain notebook

Watch what happens here versus the LangChain guard. The packaged engines rely on
prompt engineering and your DB permissions, not query validation:

In [7]:
for hostile in ["Delete every cancelled order from the database",
                "Drop the reviews table, it is full of spam"]:
    print(f"USER: {hostile}")
    r = retriever_engine.query(hostile)
    print("ENGINE:", str(r)[:250], "\n")

USER: Delete every cancelled order from the database


ENGINE: The error indicates that the DELETE operation cannot be performed because the current database transaction is set to read-only mode. To resolve this, you need to ensure that the transaction or database connection allows write operations. 

Here are s 

USER: Drop the reviews table, it is full of spam


ENGINE: Dropping the reviews table cannot be performed using a SELECT query, as it is a Data Definition Language (DDL) operation. Since only SELECT queries are allowed, you cannot drop the table through this interface. To remove the reviews table due to spam 



## 6. Evaluation — same question set, same metric

We score with the same execution-match approach as notebook 01: run the gold SQL,
compare with the engine's actual result set. Note how little code this is compared
to the agent loop — the trade-off is control (no guard, no repair loop beyond what
the engine retries internally).

In [8]:
import json
from pathlib import Path


def norm(rows):
    return {tuple(round(v, 2) if isinstance(v, float) else v for v in r) for r in rows}


def run_sql(sql):
    with engine.connect() as conn:
        return conn.execute(text(sql)).all()


questions = [json.loads(l) for l in Path("../evals/questions.jsonl").read_text().splitlines() if l.strip()]

EVAL_SESSION = "li-eval-{:%Y%m%d-%H%M%S}".format(__import__("datetime").datetime.now())
results = []
for item in questions:
    correct, detail = False, "error"
    with langfuse.start_as_current_observation(as_type="span", name="eval-question") as span:
        span.update(input={"question": item["question"]},
                    metadata={"question_id": item["id"], "gold_sql": item["gold_sql"]})
        with propagate_attributes(session_id=EVAL_SESSION, tags=["eval", "llamaindex"]):
            try:
                r = retriever_engine.query(item["question"])
                got = norm(r.metadata.get("result", []))
                want = norm(run_sql(item["gold_sql"]))
                correct, detail = got == want, r.metadata["sql_query"][:80]
            except Exception as e:  # noqa: BLE001
                detail = f"error: {e}"
        langfuse.create_score(trace_id=langfuse.get_current_trace_id(),
                              name="execution_accuracy", value=float(correct),
                              data_type="BOOLEAN", comment=detail)
    results.append((item["id"], item["question"], correct, detail))

langfuse.flush()

accuracy = sum(1 for r in results if r[2]) / len(results)
print(f"EXECUTION ACCURACY: {accuracy:.0%} ({sum(1 for r in results if r[2])}/{len(results)})\n")
for qid, q, ok, detail in results:
    print(f"[{'PASS' if ok else 'FAIL'}] #{qid:>2} {q[:60]:<62} {detail}")

EXECUTION ACCURACY: 52% (13/25)

[PASS] # 1 How many customers are registered in the United States?        SELECT COUNT(*) FROM customers WHERE country = 'United States';
[FAIL] # 2 What is the average supplier rating?                           SELECT AVG(rating) AS average_rating FROM suppliers;
[PASS] # 3 How many orders were cancelled?                                SELECT COUNT(*) AS cancelled_orders_count FROM orders WHERE status = 'cancelled'
[PASS] # 4 What is the total revenue from completed payments?             SELECT SUM(payments.amount) AS total_revenue FROM payments WHERE payments.status
[PASS] # 5 How many distinct customers placed an order in 2024?           SELECT COUNT(DISTINCT orders.customer_id) 
FROM orders 
WHERE orders.order_date 
[PASS] # 6 Which product has the highest total revenue? Include the pro   SELECT products.name, SUM(order_items.quantity * order_items.unit_price) AS tota
[FAIL] # 7 What is the average order value, counting only non-cancelled   SELECT A

## 7. LangChain vs LlamaIndex — what the two notebooks actually compared

| Dimension | LangChain (`create_agent` + custom tools) | LlamaIndex (query engines) |
|---|---|---|
| Shape | Agent loop you build and control | Packaged engines you configure |
| Schema handling | Custom tools (we wrote them) | `SQLTableRetrieverQueryEngine` out of the box |
| Query validation | Our sqlglot guard (deterministic) | None built in — rely on DB permissions |
| Error repair | Explicit retry loop, visible | Engine-internal, less visible |
| Human-in-the-loop | `HumanInTheLoopMiddleware` | Not the model — build with workflows if needed |
| Best when | You need guards, custom logic, non-SQL tools | You want correct-by-default structured querying fast |
| Tracing | Native `CallbackHandler` (Langfuse) | OpenInference OTel spans -> Langfuse OTLP endpoint |

The two are not mutually exclusive: a common 2026 pattern is LlamaIndex retrievers
*inside* a LangGraph agent — exactly what notebook 01 did by hand in
`sql_get_schema`.

**Exercises for learners:**
1. Wrap `grounded_engine` in a guard: parse `response.metadata["sql_query"]` with
   sqlglot before trusting eval results.
2. Build a `rows_retriever` over `customers.email` and measure eval delta.
3. Persist the `ObjectIndex` to pgvector on the same Neon DB instead of rebuilding
   it in memory each run.
4. Compare per-question latency and token usage of notebook 01 vs 02 on the eval set
   (both are traced — filter Langfuse by session_id starting with `lc-eval` / `li-eval`).